# الفصل 47: أسس التوليد المعزّز بالاسترجاع: RAG

> أكبر قيدين في أي LLM هما: المعرفة المحدودة بتاريخ القطع (Knowledge Cutoff) والهلوسة عند السؤال عن معلومات غير موجودة في أوزانه.

- **الجزء:** 10 — التطبيقات والوكلاء الذكية
- **الفصل:** 47
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 46: خط التقييم المستمر
- **الوسوم:** RAG، Chunking، Embeddings، Vector Search

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** faiss-cpu, numpy, sentence-transformers
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("faiss", "faiss-cpu"),
    ("numpy", "numpy"),
    ("sentence_transformers", "sentence-transformers")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch47-rag-fundamentals.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> أكبر قيدين في أي LLM هما: المعرفة المحدودة بتاريخ القطع (Knowledge Cutoff) والهلوسة عند السؤال عن معلومات غير موجودة في أوزانه.

> **بيانات الفصل**
>
> - **الجزء:** 10 — التطبيقات والوكلاء الذكية
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 46: خط التقييم المستمر


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم بناء خط RAG كامل من الصفر
- فهم التجزئة الدلالية المتقدمة
- فهم RAG


## 1. لماذا يهمّك هذا الموضوع؟

أكبر قيدين في أي LLM هما: **المعرفة المحدودة بتاريخ القطع (Knowledge Cutoff)** والهلوسة عند السؤال عن معلومات غير موجودة في أوزانه. RAG هو الحل الهندسي الأكثر نجاحًا وانتشارًا لكلتا المشكلتين، وهو العمود الفقري لمعظم تطبيقات LLM الإنتاجية اليوم (مساعدو الشركات، البحث في المستندات، أنظمة الدعم الفني).


## 2. الخريطة الذهنية


```text
RAG = الاسترجاع (Retrieval) + التوليد المُعزَّز (Augmented Generation)

سؤال المستخدم
      │
      ▼
[1. تحويل السؤال لمتجه (Embedding)]
      │
      ▼
[2. البحث عن أقرب المستندات في قاعدة متجهات]
      │
      ▼
[3. حقن المستندات المُسترجَعة كسياق في الـ Prompt]
      │
      ▼
[4. LLM يُولّد إجابة مبنية على السياق المُسترجَع]
```


🟢 **طبقة الحدس**: تخيّل RAG كـ "امتحان كتاب مفتوح" بدلاً من "امتحان من الذاكرة" — بدلاً من مطالبة النموذج بتذكّر كل شيء من تدريبه، نُعطيه "الصفحة الصحيحة من الكتاب" في لحظة السؤال.


## 3. الشرح المتدرج

### 🟡 أ) بناء خط RAG كامل من الصفر


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

class SimpleRAGPipeline:
    """
    تطبيق كامل لخط RAG أساسي — من تجزئة المستندات حتى توليد الإجابة
    """
    def __init__(self, embedding_model_name="intfloat/multilingual-e5-large",
                 generator_model=None, chunk_size=512, chunk_overlap=50):
        self.embedder = SentenceTransformer(embedding_model_name)
        self.generator = generator_model
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.index = None
        self.chunks = []
        self.chunk_metadata = []

    def chunk_document(self, text: str, source_id: str) -> list:
        """
        تجزئة النص إلى قطع متراكبة (Overlapping Chunks)
        التراكب يمنع فقدان السياق عند قطع الجملة في منتصفها
        """
        words = text.split()
        chunks = []
        step = self.chunk_size - self.chunk_overlap
        for i in range(0, len(words), step):
            chunk_words = words[i:i + self.chunk_size]
            if len(chunk_words) < 20:   # تجاهل القطع الصغيرة جدًا في النهاية
                continue
            chunks.append({
                "text": " ".join(chunk_words),
                "source_id": source_id,
                "chunk_index": i // step
            })
        return chunks

    def index_documents(self, documents: dict):
        """documents: {source_id: full_text}"""
        all_chunks = []
        for source_id, text in documents.items():
            all_chunks.extend(self.chunk_document(text, source_id))

        self.chunks = [c["text"] for c in all_chunks]
        self.chunk_metadata = all_chunks

        # E5 يتطلب بادئة "passage: " للمستندات و "query: " للأسئلة (تفصيل مهم جدًا!)
        prefixed_chunks = [f"passage: {c}" for c in self.chunks]
        embeddings = self.embedder.encode(prefixed_chunks, normalize_embeddings=True,
                                            show_progress_bar=True)

        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)   # Inner Product = Cosine (بعد التطبيع)
        self.index.add(embeddings.astype(np.float32))

    def retrieve(self, query: str, top_k=5) -> list:
        query_embedding = self.embedder.encode([f"query: {query}"], normalize_embeddings=True)
        scores, indices = self.index.search(query_embedding.astype(np.float32), top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            results.append({
                "text": self.chunks[idx],
                "score": float(score),
                "metadata": self.chunk_metadata[idx]
            })
        return results

    def build_augmented_prompt(self, query: str, retrieved_chunks: list) -> str:
        context = "\n\n".join([
            f"[مصدر {i+1}]: {chunk['text']}" for i, chunk in enumerate(retrieved_chunks)
        ])
        return f"""استخدم المعلومات التالية فقط للإجابة على السؤال. إذا لم تجد الإجابة في المعلومات المتوفرة، صرّح بذلك بوضوح ولا تختلق معلومات.

المعلومات المتاحة:
{context}

السؤال: {query}

الإجابة (مع الإشارة لرقم المصدر عند الاستشهاد):"""

    def query(self, question: str, top_k=5) -> dict:
        retrieved = self.retrieve(question, top_k)
        prompt = self.build_augmented_prompt(question, retrieved)
        answer = self.generator.generate(prompt)
        return {"answer": answer, "sources": retrieved, "prompt_used": prompt}


> 💡 **نصيحة عملية مهمة جدًا**: نماذج embedding مثل E5 تتطلب **بادئات مختلفة** للاستعلام (`query:`) والمستند (`passage:`). هذا تفصيل صغير لكن تجاهله يُنتج انخفاضًا ملموسًا في جودة الاسترجاع (قد يصل لـ 5-10 نقاط في مقاييس الاسترجاع) لأن النموذج دُرِّب تحديدًا بهذا التمييز غير المتماثل (Asymmetric Retrieval).

> ⚠️ **خطأ شائع في التجزئة (Chunking)**: تجزئة النص بحجم ثابت بدون اعتبار للبنية الدلالية (فقرات، عناوين) يؤدي أحيانًا لقطع جملة أو فكرة في المنتصف، مما يُضعف جودة التمثيل الدلالي للقطعة. الحل الأفضل: **Chunking دلالي** (Semantic Chunking) يحترم حدود الفقرات والجمل.

### 🟡 ب) التجزئة الدلالية المتقدمة


In [ ]:
import re

def semantic_chunking(text: str, max_chunk_size=512, embedder=None, similarity_threshold=0.5):
    """
    تجزئة أذكى: تُقسّم النص لجمل أولاً، ثم تُجمّع الجمل المتتالية
    المتشابهة دلاليًا في نفس القطعة، وتفصل عند نقاط التحوّل الموضوعي
    """
    sentences = re.split(r'(?<=[.!؟])\s+', text)
    if embedder is None or len(sentences) < 2:
        # fallback بسيط: تجميع حسب الحجم فقط مع احترام حدود الجمل
        chunks, current_chunk = [], ""
        for sentence in sentences:
            if len(current_chunk.split()) + len(sentence.split()) > max_chunk_size:
                chunks.append(current_chunk.strip())
                current_chunk = sentence
            else:
                current_chunk += " " + sentence
        if current_chunk:
            chunks.append(current_chunk.strip())
        return chunks

    # تجزئة دلالية حقيقية باستخدام تشابه embeddings متتالية
    sentence_embeddings = embedder.encode(sentences, normalize_embeddings=True)
    chunks, current_chunk_sentences = [], [sentences[0]]

    for i in range(1, len(sentences)):
        similarity = np.dot(sentence_embeddings[i], sentence_embeddings[i-1])
        current_length = sum(len(s.split()) for s in current_chunk_sentences)

        if similarity < similarity_threshold or current_length > max_chunk_size:
            chunks.append(" ".join(current_chunk_sentences))
            current_chunk_sentences = [sentences[i]]
        else:
            current_chunk_sentences.append(sentences[i])

    if current_chunk_sentences:
        chunks.append(" ".join(current_chunk_sentences))
    return chunks


### 🔴 ج) طبقة التعمّق: تقييم جودة RAG بمعزل عن جودة LLM

أحد أكبر التحديات المنهجية في RAG: **فصل** جودة الاسترجاع عن جودة التوليد عند تشخيص المشاكل.


In [ ]:
def evaluate_rag_components_separately(rag_pipeline, test_set: list):
    """
    إطار RAGAS المبسّط: تقييم كل مكوّن من مكونات RAG بشكل مستقل
    """
    results = {"retrieval_metrics": [], "generation_metrics": []}

    for example in test_set:
        retrieved = rag_pipeline.retrieve(example["question"], top_k=5)

        # 1. Context Precision: هل المستندات المُسترجَعة ذات صلة فعلًا؟
        relevant_retrieved = sum(
            1 for r in retrieved if r["metadata"]["source_id"] in example["relevant_source_ids"]
        )
        context_precision = relevant_retrieved / len(retrieved) if retrieved else 0

        # 2. Context Recall: هل استرجعنا كل المعلومات اللازمة للإجابة؟
        all_relevant_found = all(
            any(r["metadata"]["source_id"] == sid for r in retrieved)
            for sid in example["relevant_source_ids"]
        )
        context_recall = 1.0 if all_relevant_found else 0.0

        results["retrieval_metrics"].append({
            "precision": context_precision, "recall": context_recall
        })

        # 3. Faithfulness: هل الإجابة مبنية فعلًا على السياق المُسترجَع (وليست هلوسة)؟
        answer = rag_pipeline.query(example["question"])["answer"]
        faithfulness_prompt = f"""هل كل ادّعاء في الإجابة التالية مدعوم بالسياق المُعطى؟
السياق: {[r['text'] for r in retrieved]}
الإجابة: {answer}
أجب بـ "نعم" أو "لا" مع ذكر أي ادّعاء غير مدعوم."""
        # يتطلب LLM-as-a-Judge (من الفصل 43)
        results["generation_metrics"].append({"faithfulness_check": faithfulness_prompt})

    avg_precision = np.mean([m["precision"] for m in results["retrieval_metrics"]])
    avg_recall = np.mean([m["recall"] for m in results["retrieval_metrics"]])
    return {"avg_context_precision": avg_precision, "avg_context_recall": avg_recall}


> 💡 **نصيحة تشخيصية محورية**: إذا كانت إجابات RAG سيئة، اسأل أولًا: **"هل المشكلة في الاسترجاع أم التوليد؟"** افحص يدويًا: إذا كانت المستندات المُسترجَعة غير ذات صلة أصلًا (مشكلة Retrieval)، فلا فائدة من تحسين الـ prompt أو تبديل LLM — المشكلة في نموذج embedding أو استراتيجية التجزئة. إذا كانت المستندات صحيحة لكن الإجابة لا تزال سيئة (مشكلة Generation)، فالمشكلة في الـ prompt أو قدرة LLM نفسه.


## 4. مثال محلول خطوة بخطوة

لنحسب أثر التجزئة على الاسترجاع. مستند عربي من 1200 كلمة، ونستخدم chunk_size=512 token مع overlap=20%.

- عدد القطع ≈ $1200 / (512 \times 0.8) \approx 3$ قطع.
- إذا كان السؤال يحتاج معلومة موزعة بين القطعة 2 و3، فقد تفشل القطعة 2 وحدها.
- مع overlap 20% (≈102 token)، تظهر الجملة الحدودية في قطعتين، فترتفع فرصة الاسترجاع.

**تجربة:** اسأل 10 أسئلة، وقِس:
- Context Recall: هل القطعة الصحيحة ضمن top-k؟
- Context Precision: كم قطعة غير ذات صلة في top-k؟
- Answer Faithfulness: هل الإجابة مستندة إلى السياق؟

**النتيجة النموذجية:** chunk=512 + overlap=20% يتفوق على chunk=1024 في الأسئلة الدقيقة، لكنه يزيد عدد القطع والتكلفة. القرار يعتمد على نوع الأسئلة.


## 5. نصائح وحيل

> 💡 **استخدم بادئات embedding الصحيحة**: `query:` للسؤال و`passage:` للمستند في E5/BGE؛ تجاهلها يخفض الاسترجاع 5-10 نقاط.

> 💡 **جرّب أحجام chunk متعددة**: 256/512/1024 مع overlap 10-20%، وقِس على أسئلتك.

> 💡 **استخدم Chunking دلاليًا**: قسّم عند الفقرات/العناوين، لا عند منتصف الجملة.

> 💡 **أضف metadata**: المصدر، التاريخ، القسم؛ تسمح بفلترة وتحسين الاسترجاع.

> 💡 **افصل تقييم الاسترجاع عن التوليد**: إذا كانت القطع غير ذات صلة، لا تنفع prompt engineering.


## 6. أخطاء شائعة ومصائد

> ⚠️ **تجزئة ثابتة بدون overlap**: تقطع الجمل الحدودية وتفقد السياق.

> ⚠️ **نسيان بادئات الاستعلام/المستند**: انخفاض ملموس في الجودة.

> ⚠️ **خلط نماذج embedding**: لا تستخدم نموذجًا للفهرسة وآخر للاستعلام.

> ⚠️ **حقن كل القطع في السياق**: تكلفة عالية وتشتيت للنموذج. استخدم top-k صغيرًا + rerank.

> ⚠️ **تقييم الإجابة فقط**: إذا كانت الإجابة سيئة، لا تعرف إن السبب استرجاع أم توليد.


## 7. جدول مقارنة

| الاستراتيجية | الوصف | الأنسب لـ |
|---|---|---|
| **Fixed-size** | حجم ثابت بالكلمات/الرموز مع تراكب | بساطة سريعة، نصوص غير مُهيكَلة |
| **Recursive Character Splitting** | يحاول القطع عند `\n\n` ثم `\n` ثم المسافة كملاذ أخير | نصوص عامة متوازنة |
| **Semantic Chunking** | تجميع حسب التشابه الدلالي للجمل | دقة عالية، تكلفة حسابية أكبر |
| **Document-aware (Markdown/HTML)** | يحترم العناوين والأقسام الهيكلية | مستندات تقنية منظَّمة جيدًا |
| **Hierarchical (Parent-Child)** | قطع صغيرة للبحث + سياق أكبر (الأب) للتوليد | توازن بين دقة البحث وثراء السياق |


## 8. ملخص أفضل الممارسات

- [ ] استخدام بادئات صحيحة لنموذج embedding (query/passage)
- [ ] اختبار عدة أحجام chunk (256/512/1024) وقياس أثرها تجريبيًا
- [ ] فصل تقييم الاسترجاع عن تقييم التوليد دائمًا
- [ ] تضمين رقم/مصدر الاستشهاد في الإجابة (Traceability)
- [ ] التعامل الصريح مع حالة "لا توجد معلومة كافية" بدلاً من الهلوسة

---


## 9. أسئلة مقابلات

**س1: ما الفرق بين Context Precision وContext Recall؟**

**الإجابة النموذجية:** Context Precision يقيس نسبة القطع المسترجعة ذات الصلة (من بين كل ما استُرجع)، أي "هل ما جلبناه مفيد؟". Context Recall يقيس نسبة المعلومات اللازمة للإجابة التي نجحنا في استرجاعها، أي "هل جلبنا كل ما نحتاج؟". يجب قياسهما معًا: precision عالٍ مع recall منخفض = إجابات ناقصة، والعكس = ضوضاء.

**س2: كيف تعرف أن المشكلة في الاسترجاع لا في التوليد؟**

**الإجابة النموذجية:** افحص القطع المسترجعة يدويًا. إذا كانت لا تحتوي الإجابة، المشكلة استرجاع. إذا كانت تحتويها لكن النموذج أخطأ، المشكلة توليد/prompt. افصلهما بمقياسين.

**س3: متى تستخدم RAG بدل fine-tuning؟**

**الإجابة النموذجية:** RAG عندما تكون المعرفة متغيرة/حديثة/كبيرة ويجب الاستشهاد بالمصادر. Fine-tuning عندما تريد تغيير السلوك/الأسلوب/المهارة، أو تقليل زمن الاستجابة. كثيرًا ما يُجمعان.


## 10. تمرين عملي

1. طبّق `SimpleRAGPipeline` على 5 مستندات عربية.
2. جرّب chunk_size = 256/512/1024 مع overlap 10-20%.
3. اسأل 10 أسئلة وقِس Context Recall وPrecision يدويًا.
4. أضف reranker وقارن top-5 قبل/بعد.
5. اكتب تقريرًا: أي إعداد تختار ولماذا.


## 11. مراجع للتعمّق

- Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS.
- Karpukhin, V., et al. (2020). *Dense Passage Retrieval for Open-Domain Question Answering*. EMNLP.
- Gao, Y., et al. (2023). *Retrieval-Augmented Generation for Large Language Models: A Survey*.
- Es, S., et al. (2023). *RAGAS: Automated Evaluation of Retrieval Augmented Generation*.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التطبيقات والوكلاء الذكية / الفصل 47: أسس التوليد المعزّز بالاسترجاع: RAG.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
